In [1]:
import os
os.chdir("/home/jovyan/work/repo_conteo_gallinas")
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 2"

import torch
from ultralytics import YOLO
import pandas as pd
import shutil
import gc

import optuna


/home/jovyan/work/repo_conteo_gallinas/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MODEL_EPOCHS = 50
# MODEL_BATCH = 8
# MODEL_LR = 0.01




In [3]:
# evaluator = BinaryClassificationEvaluator(
#     labelCol="label",
#     rawPredictionCol="probability",
#     metricName="areaUnderROC"
# )


In [4]:
def objective(trial):

    model = YOLO("model/yolo26m.pt")

    # Eliminar caarpetas de resultados
    if os.path.exists("runs"):
        shutil.rmtree("runs")
    
    params = {
        "MODEL_EPOCHS": trial.suggest_int("MODEL_EPOCHS", 100, 300),
        "MODEL_BATCH": trial.suggest_int("MODEL_BATCH", 8, 12),
        "MODEL_LR": trial.suggest_float("MODEL_LR", 0.001, 0.1)
    }

    results = model.train(
        data="artifacts/full_dataset.yaml",
        imgsz=640,
        epochs=params["MODEL_EPOCHS"],
        batch=params["MODEL_BATCH"],
        lr0=params["MODEL_LR"],
        optimizer='SGD',
        device=[0, 1],
        workers=0,
        project='.',
        name='.'
    )
        
    # Lectura resultados
    df = pd.read_csv("runs/detect/results.csv")
    # epoca_final = int(df["epoch"].max())
    # tiempo_segundos = float(df["time"].max())
    # tiempo_min = tiempo_segundos/60
    # tiempo_hora = tiempo_min/60
    
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    # map50_final = float(best["metrics/mAP50(B)"])
    map5095_final = float(best["metrics/mAP50-95(B)"])
    # precision_final = float(best["metrics/precision(B)"])
    # recall_final = float(best["metrics/recall(B)"])

    # # Número de imágenes
    # imagenes = len(os.listdir("data/deteccion/imgs"))
    # imagenes_train = len(os.listdir("data/dataset/train/images"))
    # imagenes_val = len(os.listdir("data/dataset/val/images"))

    # # Información sonsolidada
    # data_list = {
    #     'epocas_inicial': params["MODEL_EPOCHS"]
    #     , 'epocas_fin': epoca_final
    #     , 'batch': params["MODEL_BATCH"]
    #     , 'learning_rate': params["MODEL_LR"]
    #     , 'optimizer': "SGD"
    #     , 'total_imagenes': imagenes
    #     , 'total_imagenes_train': imagenes_train
    #     , 'total_imagenes_val': imagenes_val

    #     , 'tiempo_minutos': tiempo_min
    #     , 'tiempo_horas': tiempo_hora
    #     , 'map50': map50_final
    #     , 'map50_95': map5095_final
    #     , "precision": precision_final
    #     , "recall": recall_final
    # }
    # data_results = pd.DataFrame([data_list])
    # data_results.to_csv(f"tmp/results_opt/result_{trial.number}.csv", index = False, encoding = "utf-8")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return map5095_final


In [5]:
study = optuna.create_study(
    direction="maximize",
    study_name="yolo_detect"
)

study.optimize(
    objective,
    n_trials=50
)

df_results = study.trials_dataframe()
df_results.to_csv(
    "tmp/results_opt_11.csv",
    index=False
)



[I 2026-11-06 06:11:49,310] A new study created in memory with name: yolo_detect


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=186, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.08532903402897103, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mod

[I 2026-11-06 06:16:06,148] Trial 0 finished with value: 0.1887 and parameters: {'MODEL_EPOCHS': 186, 'MODEL_BATCH': 11, 'MODEL_LR': 0.08532903402897103}. Best is trial 0 with value: 0.1887.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=178, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.03506343335013049, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mode

[I 2026-11-06 06:20:36,507] Trial 1 finished with value: 0.202 and parameters: {'MODEL_EPOCHS': 178, 'MODEL_BATCH': 8, 'MODEL_LR': 0.03506343335013049}. Best is trial 1 with value: 0.202.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=176, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.06561546982611578, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mod

... (7 outputs left out)

Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=11, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=168, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.09700611341784775, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mod

[I 2026-11-06 06:45:10,739] Trial 7 finished with value: 0.18028 and parameters: {'MODEL_EPOCHS': 168, 'MODEL_BATCH': 11, 'MODEL_LR': 0.09700611341784775}. Best is trial 4 with value: 0.20554.


Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=205, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0652183588904723, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=mode

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

